# detr-lens — Interactive Exploration

Explore attention maps from Deformable DETR and RT-DETR.

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
%matplotlib inline

In [ ]:
from models.rtdetr import RTDETRWrapper
from models.deformable_detr import DeformableDETRWrapper

rt = RTDETRWrapper()
rt.load()
print('RT-DETR loaded')

In [ ]:
img = Image.open('../data/coco/val2017/000000000139.jpg').convert('RGB')
out = rt.forward(img)
print(f"Detections: {len(out['boxes'])}")
print(f"Cross-attn layers: {len(out['cross_attn_weights'])}")
print(f"Attn shape: {out['cross_attn_weights'][0].shape}")
print(f"Sampling locs shape: {out['sampling_offsets'][0].shape}")

In [ ]:
from viz.heatmap import render_heatmap, render_rollout, render_head_diversity

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

hm = render_heatmap(img, out, layer_idx=-1, query_idx=0)
axes[0].imshow(hm); axes[0].set_title('Cross-attn heatmap (last layer)'); axes[0].axis('off')

ro = render_rollout(img, out, query_idx=0)
axes[1].imshow(ro); axes[1].set_title('Attention rollout'); axes[1].axis('off')

dv = render_head_diversity(out)
axes[2].imshow(dv); axes[2].set_title('Head diversity'); axes[2].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
from viz.offset_grid import render_sampling_offsets, render_per_head_grid

# Show all heads' sampling locations
offsets = render_sampling_offsets(img, out, layer_idx=-1, query_idx=0)
plt.figure(figsize=(10, 6))
plt.imshow(offsets)
plt.title('Deformable sampling offsets — all heads')
plt.axis('off')
plt.show()

In [ ]:
# Per-head grid
grid = render_per_head_grid(img, out, layer_idx=-1, query_idx=0)
plt.figure(figsize=(14, 7))
plt.imshow(grid)
plt.title('Sampling locations per head')
plt.axis('off')
plt.show()

In [ ]:
from viz.side_by_side import render_side_by_side

# Load Deformable DETR for comparison
dd = DeformableDETRWrapper()
dd.load()
out_dd = dd.forward(img)

comp = render_side_by_side(img, out_dd, out, mode='heatmap', query_idx=0)
plt.figure(figsize=(16, 5))
plt.imshow(comp)
plt.title('Deformable DETR vs RT-DETR — heatmap')
plt.axis('off')
plt.show()

In [ ]:
# Layer sweep for RT-DETR
n_layers = len(out['cross_attn_weights'])
fig, axes = plt.subplots(1, n_layers, figsize=(n_layers * 3.5, 3))
for i in range(n_layers):
    h = render_heatmap(img, out, layer_idx=i, query_idx=0)
    axes[i].imshow(h)
    axes[i].set_title(f'Layer {i}')
    axes[i].axis('off')
plt.suptitle('RT-DETR cross-attention across decoder layers', y=1.02)
plt.tight_layout()
plt.show()